In [ ]:
import os
import pandas as pd
import torch
import numpy as np


In [ ]:
output_dir = "data/homecredit"
os.makedirs(output_dir, exist_ok=True)


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


def load_dataset(path):
    data = pd.read_csv(path)

    cat_features = ["NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE", "ORGANIZATION_TYPE"]

    all_features = [
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "AMT_GOODS_PRICE",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
        "EXT_SOURCE_1",
        "NAME_EDUCATION_TYPE",
        "DAYS_EMPLOYED",
        "NAME_INCOME_TYPE",
        "ORGANIZATION_TYPE",
        "REGION_POPULATION_RELATIVE",
        "age",
    ]

    X = data[all_features]
    y = data["TARGET"].values  # 0 loan was repaid, 1 loan was not repaid
    s_raw = data["CODE_GENDER"].values
    s = (s_raw == "M").astype(int)  # 0 is female, 1 is male

    L = X["AMT_CREDIT"].values
    A = X["AMT_ANNUITY"].values

    num_features = [col for col in all_features if col not in cat_features]

    num_transformer = Pipeline(
        [("imputer", SimpleImputer(strategy="mean")), ("scaler", MinMaxScaler())]
    )

    cat_transformer = Pipeline(
        [("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))]
    )

    # Combine transformers
    preprocessor = ColumnTransformer(
        [("num", num_transformer, num_features), ("cat", cat_transformer, cat_features)]
    )

    # Fit-transform the data
    X_processed = preprocessor.fit_transform(X)

    # Recover feature names
    num_names = num_features
    cat_names = (
        preprocessor.named_transformers_["cat"]
        .named_steps["onehot"]
        .get_feature_names_out(cat_features)
    )
    feature_names = list(num_names) + list(cat_names)

    # Convert to DataFrame
    X_df = pd.DataFrame(
        X_processed, columns=feature_names
    )  # currently used only for scores
    y = 1 - y

    return X_df, y, s, L, A


In [ ]:
from sklearn.model_selection import train_test_split

# Step 1: Load data
X_df, y, S, L, A = load_dataset()

X_aug = X_df.copy()
X_aug["S"] = S

# Step 2: Train/test split
X_train, X_test, y_train, y_test, S_train, S_test, L_train, L_test, A_train, A_test = (
    train_test_split(X_aug, y, S, L, A, test_size=0.3, random_state=42, stratify=y)
)




In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# Define your model
class SmallNet(nn.Module):
    def __init__(self, input_dim):
        super(SmallNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(32, 1),  # outputs logits
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
x_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(
    1
)  # make it (N, 1)
L_train_tensor = torch.tensor(L_train, dtype=torch.float32)
A_train_tensor = torch.tensor(A_train, dtype=torch.float32)
s_train_tensor = torch.tensor(S_train, dtype=torch.long)

x_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
L_test_tensor = torch.tensor(L_test, dtype=torch.float32)
A_test_tensor = torch.tensor(A_test, dtype=torch.float32)
s_test_tensor = torch.tensor(S_test, dtype=torch.long)


In [ ]:
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
input_dim = X_train.shape[1]
model = SmallNet(input_dim)

criterion = nn.BCEWithLogitsLoss()  # since output is logits
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)
    avg_loss = epoch_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(x_test_tensor)
    probs_test = torch.sigmoid(logits)
    preds = (probs_test >= 0.5).float()
    acc = (preds.eq(y_test_tensor)).sum().item() / y_test_tensor.size(0)
    print("Test accuracy:", acc)

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(x_train_tensor)
    probs_train = torch.sigmoid(logits)
    preds = (probs_train >= 0.5).float()
    acc = (preds.eq(y_train_tensor)).sum().item() / y_train_tensor.size(0)
    print("Test accuracy:", acc)

In [ ]:
# %%
# Save Home Credit train/test datasets with predicted probabilities

model.eval()
with torch.no_grad():
    # Train probabilities
    train_logits = model(x_train_tensor)
    train_probs = torch.sigmoid(train_logits).cpu().numpy().flatten()

    # Test probabilities
    test_logits = model(x_test_tensor)
    test_probs = torch.sigmoid(test_logits).cpu().numpy().flatten()

# Recreate DataFrames
train_df = X_train.copy()
train_df=train_df.add_prefix("preprocessed_")
train_df=train_df.rename(columns={"preprocessed_S":"G"})
train_df["y"] = y_train
train_df["L"] = L_train
train_df["A"] = A_train
train_df["prob_Y"] = train_probs


test_df = X_test.copy()
test_df=test_df.add_prefix("preprocessed_")
test_df=test_df.rename(columns={"preprocessed_S":"G"})
test_df["y"] = y_test
test_df["L"] = L_test
test_df["A"] = A_test
test_df["prob_Y"] = test_probs


# Paths
train_path = os.path.join(output_dir, "homecredit_train_with_prob.csv")
test_path = os.path.join(output_dir, "homecredit_test_with_prob.csv")

# Save
# train_df.to_csv(train_path, index=False)
# test_df.to_csv(test_path, index=False)

print(f"Saved train set to {train_path}")
print(f"Saved test set to {test_path}")